# Classical QSAR Baselines

## Scientific objective
Train endpoint-specific dummy, logistic regression, random forest, SVM, gradient boosting, and XGBoost controls under the same primary scaffold split.

## Inputs
- Feature matrix and modeling records
- Training/model configs

## Expected outputs
- `models/qsar/*_estimators.joblib`
- `results/metrics/qsar_baselines.csv`
- endpoint/model predictions
- `reports/qsar_training_progress.json`

## Dependencies
scikit-learn, XGBoost optional, joblib

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
The smoke profile caps training rows and validates the pipeline; the full profile is required for manuscript-grade repeated estimates and confidence intervals.

## Validation checks
The logistic preflight verifies both fitting and BLAS-independent validation prediction before the complete estimator registry is executed.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Model comparison before calibration is preliminary. Hyperparameters and thresholds must never be selected on the test partition.

## Next notebook
[11_single_task_neural_models.ipynb](./11_single_task_neural_models.ipynb)


In [1]:
import os

# These variables must be set before importing NumPy, SciPy, scikit-learn,
# XGBoost, Matplotlib, or PyTorch in this kernel.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["BLIS_NUM_THREADS"] = "1"
os.environ["MPLBACKEND"] = "Agg"

from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})


{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from sklearn.base import clone

from toxicity_screening.evaluation import (
    build_qsar_estimators,
    predict_qsar_probability,
)
from toxicity_screening.pipeline import (
    _load_model_matrix,
    _sample_training_rows,
)

X, feature_index, preflight_records = _load_model_matrix(ROOT)
preflight_records = preflight_records.merge(
    feature_index,
    on="molecule_id",
    how="left",
    validate="many_to_one",
)
preflight_endpoint = preflight_records.loc[
    (preflight_records["endpoint"] == "herg_blockade")
    & preflight_records["label"].notna()
].copy()

preflight_train = preflight_endpoint.loc[
    preflight_endpoint["scaffold_split"] == "train"
]
preflight_train = _sample_training_rows(
    preflight_train,
    PROFILE_CONFIG["sample_cap_per_endpoint"],
    SEED,
)
preflight_validation = preflight_endpoint.loc[
    preflight_endpoint["scaffold_split"] == "validation"
]

preflight_x = X[preflight_train["row"].astype(int)]
preflight_y = preflight_train["label"].astype(int).to_numpy()
preflight_validation_x = X[
    preflight_validation["row"].astype(int)
]

print("Logistic preflight matrix:", preflight_x.shape, preflight_x.dtype)
print("Logistic preflight class counts:", np.bincount(preflight_y))
print(
    "Logistic validation matrix:",
    preflight_validation_x.shape,
    preflight_validation_x.dtype,
)

preflight_model = clone(
    build_qsar_estimators(SEED)["logistic_regression"]
)
print(preflight_model)
preflight_model.fit(preflight_x, preflight_y)
print("Logistic regression fit passed.")

preflight_probability = predict_qsar_probability(
    preflight_model,
    preflight_validation_x,
    model_name="logistic_regression",
)

assert preflight_probability.shape == (
    len(preflight_validation),
)
assert np.isfinite(preflight_probability).all()
assert (
    (preflight_probability >= 0.0)
    & (preflight_probability <= 1.0)
).all()

print(
    "Safe logistic validation prediction passed:",
    preflight_probability.shape,
    float(preflight_probability.min()),
    float(preflight_probability.max()),
)


Logistic preflight matrix: (8971, 2048) uint8
Logistic preflight class counts: [4602 4369]
Logistic validation matrix: (2095, 2048) uint8
SafeLogisticPipeline(steps=[('scale', StandardScaler(with_mean=False)),
                            ('model',
                             LogisticRegression(class_weight='balanced',
                                                max_iter=2000,
                                                random_state=20260723,
                                                solver='liblinear'))])
Logistic regression fit passed.
Safe logistic validation prediction passed: (2095,) 4.4711696235413735e-13 0.9999999999929392


In [3]:
from toxicity_screening.pipeline import train_qsar_baselines

QSAR_PROGRESS_PATH = ROOT / "reports" / "qsar_training_progress.json"
print("QSAR progress report:", QSAR_PROGRESS_PATH)
print("The file is updated before and after every model-fit and prediction stage.")

metrics = train_qsar_baselines(ROOT, PROFILE)
display(
    metrics.sort_values(
        ["endpoint", "partition", "pr_auc"],
        ascending=[True, True, False],
    ).head(30)
)


QSAR progress report: D:\Dropbox\Work\Learning\Python\toxicity_screening_project\reports\qsar_training_progress.json
The file is updated before and after every model-fit and prediction stage.
[QSAR] matrix_load_started
[QSAR] matrix_load_completed molecules=25583 records=50612 features=2048
[QSAR] estimator_registry_started
[QSAR] estimator_registry_completed models=['dummy', 'logistic_regression', 'random_forest', 'svm', 'gradient_boosting', 'xgboost']
[QSAR] endpoint_started endpoint=herg_blockade
[QSAR] model_fit_started endpoint=herg_blockade model=dummy train_rows=8971 features=2048
[QSAR] model_fit_completed endpoint=herg_blockade model=dummy
[QSAR] prediction_started endpoint=herg_blockade model=dummy partition=validation rows=2095 backend=native_predict_proba
[QSAR] prediction_completed endpoint=herg_blockade model=dummy partition=validation backend=native_predict_proba
[QSAR] prediction_started endpoint=herg_blockade model=dummy partition=test rows=1883 backend=native_predict_

,endpoint,model,split_strategy,partition,n,positive_prevalence,threshold,roc_auc,pr_auc,mcc,...,f1,brier,ece,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp
55,SR-ARE,svm,global_scaffold,test,847,0.205431,0.5,0.814644,0.565105,0.327128,...,0.334802,0.126780,0.038675,0.400734,0.137931,0.382514,658,15,136,38
59,SR-ARE,xgboost,global_scaffold,test,847,0.205431,0.5,0.788334,0.543534,0.341853,...,0.321101,0.131215,0.060527,0.420480,0.293103,0.356061,664,9,139,35
53,SR-ARE,random_forest,global_scaffold,test,847,0.205431,0.5,0.788390,0.526115,0.238426,...,0.221154,0.133435,0.039224,0.424452,0.097701,0.352941,662,11,151,23
57,SR-ARE,gradient_boosting,global_scaffold,test,847,0.205431,0.5,0.744565,0.478851,0.312202,...,0.355372,0.141502,0.069271,0.478235,0.120690,0.303226,648,25,131,43
51,SR-ARE,logistic_regression,global_scaffold,test,847,0.205431,0.5,0.642154,0.304688,0.167890,...,0.361858,0.275349,0.273922,2.276900,0.000000,0.241087,512,161,100,74
49,SR-ARE,dummy,global_scaffold,test,847,0.205431,0.5,0.500000,0.205431,0.000000,...,0.000000,0.166324,0.055636,0.518949,0.000000,0.205431,673,0,174,0
54,SR-ARE,svm,global_scaffold,validation,680,0.164706,0.5,0.748609,0.360090,0.198995,...,0.246753,0.126164,0.033387,0.399380,0.000000,0.265306,545,23,93,19
52,SR-ARE,random_forest,global_scaffold,validation,680,0.164706,0.5,0.736324,0.359233,0.153357,...,0.172662,0.124993,0.037055,0.395892,0.026786,0.257143,553,15,100,12
58,SR-ARE,xgboost,global_scaffold,validation,680,0.164706,0.5,0.714820,0.336336,0.202686,...,0.262500,0.130343,0.045518,0.419839,0.008929,0.233333,541,27,91,21
56,SR-ARE,gradient_boosting,global_scaffold,validation,680,0.164706,0.5,0.708910,0.333685,0.206844,...,0.277108,0.134719,0.069206,0.452388,0.000000,0.218009,537,31,89,23


In [4]:
for endpoint in metrics.endpoint.unique():
    test_models = metrics[
        (metrics.endpoint == endpoint)
        & (metrics.partition == "test")
    ]
    assert "dummy" in set(test_models.model)
    assert test_models.pr_auc.notna().any()


In [5]:
from toxicity_screening.pipeline import repeated_scaffold_qsar_evaluation

runs, repeated_summary = repeated_scaffold_qsar_evaluation(ROOT, PROFILE)
display(repeated_summary)


,endpoint,model,metric,runs,mean,std,ci95_lower,ci95_upper
0,SR-ARE,logistic_regression,roc_auc,5,0.660621,0.014490,0.642628,0.678613
1,SR-ARE,logistic_regression,pr_auc,5,0.373428,0.009154,0.362062,0.384794
2,SR-ARE,logistic_regression,mcc,5,0.210169,0.027390,0.176159,0.244179
3,SR-ARE,logistic_regression,balanced_accuracy,5,0.609442,0.013012,0.593286,0.625599
4,SR-ARE,logistic_regression,sensitivity,5,0.433888,0.031165,0.395191,0.472584
...,...,...,...,...,...,...,...,...
151,herg_blockade,random_forest,brier,5,0.163824,0.005878,0.156526,0.171122
152,herg_blockade,random_forest,ece,5,0.054119,0.013611,0.037220,0.071019
153,herg_blockade,random_forest,nll,5,0.496361,0.013778,0.479253,0.513469
154,herg_blockade,random_forest,recall_at_precision_0.80,5,0.707921,0.070661,0.620184,0.795658


### Completion gate
Confirm that the declared artifacts exist before continuing to `11_single_task_neural_models.ipynb`.
